# Krussel Smith Implementation

Here is where I give a whack at getting Krussel Smith going. I am trying not to crash out (interesting strategy, let's see how it plays out).

In [1]:
# making sure necessary packages are installed
using Pkg
Pkg.activate(".")   # so you're using the same packages as me
Pkg.instantiate()

  Activating project at `C:\VAASAVI\Dropbox\Education\OSU\Ongoing_Research\Populism\political-polarization\p`


In [2]:
using JLD2
using Random
using Printf
using LinearAlgebra: dot
using StatsBase: countmap

for file in [
    "src/ModelTypes.jl",
    "src/Compute.jl",
    "src/ModelFunctions.jl",
    "src/DistrTools.jl",
    "src/EGM.jl",
    "src/Solvers.jl",
    "src/SteadyState.jl",
    "src/Predict.jl"]
    include(file)
    @printf("Loaded %s\n", basename(file))
end

using .ModelTypes
using .ModelFunctions
using .Compute
using .EGM
using .DistrTools
using .Solvers
using .SteadyState
using .Predict

Loaded ModelTypes.jl
Loaded Compute.jl
Loaded ModelFunctions.jl
Loaded DistrTools.jl
Loaded EGM.jl
Loaded Solvers.jl
Loaded SteadyState.jl
Loaded Predict.jl


In [4]:
# ─── Frequency-invariant parameters ───
const α::Float64 = 0.36
const σ::Float64 = 2
const ϕ::Float64 = 0
const μ_l::Float64 = 0
const μ_z::Float64 = 0

# grid sizes and parameters
const na::Int64 = 100; 
const nl::Int64 = 15;
const nz::Int64 = 5;
const nk::Int64 = 25;

const a_l::Float64 = 0;
const a_h::Float64 = 100;

# kgrid
const kL::Float64 = 6; const kH::Float64 = 16; # this could be informed by steady states, but for now we do it this way
Kgrid = collect(range(kL, kH, length = nk));

# ─── Frequency switch: read from environment, default to "annual" ───
const freq = get(ENV, "FREQ", "quarterly")   # default to quarterly if not set

if freq == "quarterly"
    const β::Float64   = 0.99
    const δ::Float64   = 0.025
    const ρ_l::Float64 = 0.982      # 0.9345^(1/4), STY persistence quarterly
    const σ_l::Float64 = 0.127      # STY innovation std, quarterly
    const ρ_z::Float64 = 0.976
    const σ_z::Float64 = 0.007
    
    const Kfore_start = repeat([0.0 1.0], nz, 1) # a guess bc I don't have an eqm yet
elseif freq == "annual"
    const β::Float64   = 0.96
    const δ::Float64   = 0.06
    const ρ_l::Float64 = 0.9345     # Storesletten-Telmer-Yaron
    const σ_l::Float64 = 0.247      # = sqrt(0.061), STY persistent innovation variance σ²_η
    const ρ_z::Float64 = 0.909      # Khan-Thomas 2013
    const σ_z::Float64 = 0.014
    
    const Kfore_start = [0.102898 0.946118;  #taken from previous run
			0.112504 .944115;
			0.121485 0.942448;
			0.131148 0.940548;
			0.139579 0.939373]
else
    error("FREQ must be \"annual\" or \"quarterly\", got \"$freq\"")
end

5×2 Matrix{Float64}:
 0.0  1.0
 0.0  1.0
 0.0  1.0
 0.0  1.0
 0.0  1.0

In [5]:

grid_range = 2.575;

π_l, lgrid = getTauchen(nl,  μ_l, σ_l, ρ_l, grid_range);
π_z, zgrid= getTauchen(nz,  μ_z, σ_z, ρ_z, grid_range);

stationary_l = stationary(π_l);
const lagg::Float64 = dot(stationary_l, lgrid);
 
agrid = logspace(a_l, a_h, na);
amu = collect(range(a_l, a_h, length=na*10));

const kL::Float64 = 6; const kH::Float64 = 12; # this could be informed by steady states, but for now we do it this way
Kgrid = collect(range(kL, kH, length = nk));

In [6]:
const np::Int64 = 10; # number of policies
const pol_l::Float64 = 0;
const pol_h::Float64 = 1;

τ_grid = range(pol_l, pol_h, length = np);
η_grid = range(pol_l, pol_h, length = np);

captax = repeat([0], outer = nl);

η = 0.05; τ = 0;

Okay, so the goal here is that there are forecasts across moments of the aggregate capital distribution, which means that I have to send new prices to the household across capital moments and solve each household problem at each aggregate capital moment; then I simulate, check the accuracy of the forecasts with the actual responses to a TFP shock path, and then update my forecasts. 

What this means is that every aggregate capital moment generates a price, which leads to the solution of the household problem for each aggregate capital moment with uncertainty over TFP. To do that, I need to make a couple changes: 

0. Choosing which steady state to start from--I need to import that stationary distribution. 
1. Set up a forecasting rule. Here, I'll just set it to something like $\log(K') = 0.05 a + .95\log(K)$ for each transition z -> z' (so this is nz^2 forecasting rules)
2. Simulating a shock path: I want to keep this constant, so it's setting the RNG seed so that it spits out the same z path each time.  
3. Then running regressions and updating--because I have effectively 4 rules, this should have several periods. 

In [7]:
const NT = 5000; #three thousand periods for sampling
const rnseed = 1234567;

zt = simz(NT, nz, rnseed, π_z);

Now to do the actual solve:

In [8]:
V = nothing
EV = nothing
G = nothing  
C = nothing
CI = nothing
LI = nothing
μ = nothing

foredist = 10;
const dTol = 1e-3;
const vTol = 1e-6;


r_vals = zeros(nk, nz); w_vals = zeros(nk, nz); λ_vals = zeros(nk, nz); 

for ik = 1:nk,  iz = 1:nz
    r_vals[ik, iz] = calcr(α, δ, Kgrid[ik], η, zgrid[iz])
    w_vals[ik, iz] = calcw(α, Kgrid[ik], η, zgrid[iz])
    denom = dot((w_vals[ik, iz] .* lgrid).^(1 - τ), stationary(π_l))
    tot_inc = w_vals[ik, iz] * dot(lgrid, stationary(π_l))
    λ_vals[ik, iz] = tot_inc / denom
end

const params = ModelParams(α, β, δ, σ, ϕ, agrid, 
    lgrid, zgrid, π_l, π_z, amu, Kgrid);

const policies = ProposedPolicies(η, τ, captax);

const prices = ImpliedRegimeParams_KS(λ_vals, r_vals, w_vals)

# init V:

V0 = zeros(nk,nz,nl,na); V  = zeros(nk,nz,nl,na);
EV = zeros(nk,nz,nl,na); G  = zeros(nk,nz,nl,na); 
G0 = zeros(nk, nz, nl, na); C  = zeros(nk,nz,nl,na);

for ik = 1:nk,  iz = 1:nz, il = 1:nl, ia = 1:na
    kval = agrid[ia];
    yval = (1 + r_vals[ik, iz]*(1-captax[il]))*kval + w_vals[ik, iz]*lgrid[il] - r_vals[ik, iz]*ϕ;
    ymin = max(1e-10, yval);
    V0[ik, iz, il, ia] = log(ymin);
    G0[ik, iz, il, ia] = agrid[ia];
end


In [9]:
print(policies)

ProposedPolicies(0.05, 0.0, [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

In [10]:
Kfore, Kt = run_KS(V, V0, G, G0, C, params, policies, prices,
                zt, Kfore_start, vTol, dTol, verbose = true)

@printf("K range for (%4.2f, %4.2f): %2.4f, %2.4f\n", 
    η, τ,
    minimum(Kt), maximum(Kt))

Solving Household Problem...
	Converged in 1 iters, dist = 10.000000
	Simulating period 2500 of 5000
	Simulating period 5000 of 5000

Forecast rules:  log K' = a + b·log K
──────────────────────────────────────────────────────────
  z-state            a           b        R²        n
──────────────────────────────────────────────────────────
  1                NaN         NaN       NaN       50
  2                NaN         NaN       NaN     1156
  3                NaN         NaN       NaN      693
  4                NaN         NaN       NaN     1888
  5                NaN         NaN       NaN      713
──────────────────────────────────────────────────────────
Outer  1 | foredist = NaN | R² = [NaN, NaN, NaN, NaN, NaN] | counts = [50, 1156, 693, 1888, 713]

Hit maxout=100 without converging. foredist = NaN
Maxout at policy (η, τ) = (0.05, 0.00)
K range for (0.05, 0.00): 0.0000, 9.0000


In [9]:
results = Dict{Tuple{Float64,Float64}, NamedTuple}()

# storing each stationary equilibrium
results[(η, τ)] = (
    Kfore,
    V = V,
    G = G,
    C = C,
    Kt = Kt,
    zt = zt,
    policies = policies
);

In [10]:
@save "../d/KS_solves.jld2" results params zt

In [11]:
print(π_z)

[0.8370276127980094 0.16294901698975173 2.3370211836626353e-5 4.022338018216942e-13 0.0; 0.03395354720350539 0.8628268622111023 0.10321285959633375 6.730989008829624e-6 4.973799150320701e-14; 1.7971484058070882e-6 0.061229388312376475 0.8775376290784354 0.061229388312376565 1.7971484057577314e-6; 4.977489407637338e-14 6.7309890088411185e-6 0.10321285959633363 0.8628268622111024 0.03395354720350541; 5.951816699063212e-25 4.02223261070338e-13 2.337021183662143e-5 0.1629490169897517 0.8370276127980094]